### クロスバリデーション＋グリッドサーチ

#### ライブラリとデータの読み込み

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import os
import re
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

#表情データの読み込みとデータフレームに格納
folder_path = Path('../../../filtered_face_data_0.95/')

file_list = folder_path.glob('*.csv')

file_path_list = [str(p) for p in file_list]

df = pd.DataFrame(file_path_list, columns=['filepath'])

files_df = df
files_df['filename'] = df['filepath'].apply(lambda p: Path(p).name)
files_df['ID'] = files_df['filename'].str.extract(r'ID(\d+)')
files_df['ID'] = files_df['ID'].astype(int)

#患者の疾患有無
labels_df = pd.read_csv(r"C:\Users\robotics\proj\Research\code\voice\delirium.csv")
#患者と疾患有無を結合
file_with_labels_df = pd.merge(files_df, labels_df, on='ID', how='left')
file_with_labels_df.drop(columns=['Sex'], inplace=True)

print(file_with_labels_df.columns)

Index(['filepath', 'filename', 'ID', 'Delirium'], dtype='object')


#### 前処理の関数(応答部分全体の最大値)

In [2]:
def answer_max(file_path: list, data: pd.Series, file_lengths: list):
    df = pd.DataFrame()
    for f, a in zip(file_path, data):
        #ファイル読み込み、ファイル名切り出し
        df_file = pd.read_csv(f)
        current_filename = os.path.basename(f)
        
        
        df_file['Delirium'] = a
        df_file['filename'] = current_filename

        df_file['label'] = df_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_sort_df = df_file[df_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_sort_df[confidence].index
        df_new_df = df_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_new_df.columns if col.startswith("AU")] + ["Delirium", "filename"]
        df_final_df = df_new_df[columns_to_extract]

        #print(df_test_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_final_df = df_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_test_final_df)

        #数値型に変更
        #df_test_final_df['filename'] = pd.to_numeric(df_test_final_df['filename'], errors='coerce')
        #print(df_test_final_df)

        #質問ごとの最大値を計算
        df_max_df = df_final_df.groupby('label').max()
        df_max_df = df_max_df.reset_index(drop=True)
        #print(df_test_max_df)
        
        file_lengths.append(len(df_max_df))

        df = pd.concat([df, df_max_df], ignore_index=True)

    return file_lengths, df



#### 応答部分全体(フレーム数を考慮)

In [ ]:
def answer_max(file_path: list, data: pd.Series, file_lengths: list):
    df = pd.DataFrame()
    for f, a in zip(file_path, data):
        #ファイル読み込み、ファイル名切り出し
        df_file = pd.read_csv(f)
        current_filename = os.path.basename(f)
        #df_file = df_file.dropna(how='all', axis=1)
        
        df_file['Delirium'] = a
        df_file['filename'] = current_filename

        df_file['label'] = df_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_sort_df = df_file[df_file['label'].str.match('^answer(10|[1-9])$', na=False)]

        #フレーム数の特徴量を追加


        columns_to_extract = ["label"] + [col for col in df_sort_df.columns if col.startswith("AU")] + ["Delirium", "filename"]
        df_final_df = df_sort_df[columns_to_extract]

        #print(df_test_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_final_df = df_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_test_final_df)
        

        #数値型に変更
        #df_test_final_df['filename'] = pd.to_numeric(df_test_final_df['filename'], errors='coerce')
        #print(df_test_final_df)

        #質問ごとの平均値を計算
        df_final_df['count'] = df_final_df.groupby('label')['label'].transform('size')
        df_final_df['filename_num'] = df_final_df['filename'].astype(str).str.extract(r'(\d+)').astype(float).astype(int)
        #print(df_final_df.columns)
        df_final_df = df_final_df.drop(columns=['filename_num'], errors='ignore')
        print(df_final_df.info())
        

        df_max_df = df_final_df.groupby('label').max()
        df_max_df = df_max_df.reset_index(drop=True)
        
        
        file_lengths.append(len(df_max_df))

        df = pd.concat([df, df_max_df], ignore_index=True)

    return file_lengths, df



In [42]:
def answer_max(file_path: list, data: pd.Series, file_lengths: list):
    df = pd.DataFrame()
    for f, a in zip(file_path, data):
        #ファイル読み込み、ファイル名切り出し
        df_file = pd.read_csv(f)
        current_filename = os.path.basename(f)
        
        
        df_file['Delirium'] = a
        df_file['filename'] = current_filename

        df_file['label'] = df_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_sort_df = df_file[df_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_sort_df[confidence].index
        df_new_df = df_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_new_df.columns if col.startswith("AU")] + ["Delirium", "filename"]
        df_final_df = df_new_df[columns_to_extract]

        #print(df_test_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_final_df = df_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_test_final_df)

        #数値型に変更
        #df_test_final_df['filename'] = pd.to_numeric(df_test_final_df['filename'], errors='coerce')
        #print(df_test_final_df)

        #質問ごとの最大値を計算
        df_max_df = df_final_df.groupby('label').max()
        df_max_df = df_max_df.reset_index(drop=True)
        #print(df_test_max_df)
        
        file_lengths.append(len(df_max_df))

        df = pd.concat([df, df_max_df], ignore_index=True)

    return file_lengths, df



#### データ分割, グリッドサーチ , 学習, 評価

In [3]:
#学習・テストデータとなるファイルの分割
X_files = file_with_labels_df.drop(columns='Delirum', errors='ignore')
#print(X_files)
y_labels = file_with_labels_df.drop(columns=['filepath', 'filename', 'ID'], errors='ignore')
#rint(y_labels)

#層化５分割交差検証
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(skf.split(X_files, y_labels)):
    X_train_sort, X_test_sort = X_files.iloc[train_index], X_files.iloc[test_index]
    y_train_sort, y_test_sort = y_labels.iloc[train_index], y_labels.iloc[test_index]

    train_file_path = [f for f in X_train_sort['filepath']]
    #print(type(train_file_path), type(y_train_sort['Delirium']))
    test_file_path = [f for f in X_test_sort['filepath']]
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()
    train_file_lengths = []
    test_file_lengths = []
    #print(type(test_file_lengths))

    #学習データ
    train_file_lengths, train_df = answer_max(
        train_file_path,
        y_train_sort['Delirium'],
        train_file_lengths)
    train_df['filenumber'] = train_df['filename'].str.extract(r'(\d+)')
    train_df = train_df.drop(columns=['filename'], errors='ignore')
    print(train_df)
    #テストデータ
    test_file_lengths, test_df = answer_max(
        test_file_path,
        y_test_sort['Delirium'],
        test_file_lengths
    )

    #グリッドサーチのための学習・検証データの用意とテストデータの用意
    y_train_grid = train_df['Delirium']
    y_test_grid = test_df['Delirium']

    columns_to_delete = ['Delirium']
    columns_to_delete2 = ['Delirium', 'filename']

    X_train_grid = train_df.drop(columns=columns_to_delete, errors='ignore')
    X_test_grid = test_df.drop(columns=columns_to_delete2, errors='ignore')

    #患者ごとに分割するための変数を用意し、学習データからfilenumberを削除
    patient_groups = X_train_grid['filenumber'].values
    X_train_model = X_train_grid.drop(columns=['filenumber']).copy()
    #モデル・グリッドサーチの定義
    model = xgb.XGBClassifier(eval_metric='logloss')
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    param_grid = {
            'learning_rate': [0.01, 0.1],
            'max_depth': [3, 5, 7, 10],
            'min_child_weight': [1, 3, 5],
            'subsample': [0.5, 0.7],
            'colsample_bytree': [0.5, 0.7],
            'n_estimators' : [100, 200, 500],
            'objective': ['reg:squarederror']
        }
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=cv, n_jobs=-1)

    #グリッドサーチ
    grid_search.fit(X_train_model, y_train_grid, groups=patient_groups)

    print(f"最適なハイパーパラメータ：{grid_search.best_params_}")
    print(f"CVでのmax精度(Accuracy):{grid_search.best_score_:.4f}")

    #行と列の表示制限を解除（または十分に大きな値に設定）
    pd.set_option('display.max_rows', None)     # すべての行を表示
    pd.set_option('display.max_columns', None)  # すべての列を表示
    pd.set_option('display.width', 1000)        # 表示幅を広くする

    #結果をDataFrameに変換し表示
    results_df = pd.DataFrame(grid_search.cv_results_)

    print("--- グリッドサーチの全試行結果 ---")
    print(results_df)

    #表示設定を元に戻す (推奨)
    #グローバルな設定を元に戻し、他の処理に影響を与えないようにします
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')


    #テストデータでの性能評価
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_grid)
    final_accuracy = accuracy_score(y_test_grid, y_pred)

    print(f"テストデータでの正解率: {final_accuracy:.4}\n")
    #print(y_pred)

    file_predictions = []
    start_idx = 0
    for length in test_file_lengths:
        end_idx = start_idx + length
        predictions_for_this_file = y_pred[start_idx:end_idx]
        majority_vote = mode(predictions_for_this_file)[0]
        file_predictions.append([majority_vote, start_idx, end_idx])
        start_idx = end_idx

    first_column_list = [item[0] for item in file_predictions]

    #print(first_column_list)

    y_test_files = X_test_sort['Delirium'].values

    #print(type(y_test_files))

    y_test_files_copy = X_test_sort

    y_test_files_copy = y_test_files_copy.assign(Predictions=first_column_list)

    print("各テストファイルごとの分類結果\n")
    print(y_test_files_copy.drop(columns=['filepath'], errors='ignore'))

    correct_files = np.sum(np.array(first_column_list) == y_test_files)
    total_files = len(X_test_sort)
    file_accuracy = correct_files/ total_files

    scores = []
    scores.append(file_accuracy)
    print("\n")
    print(f"正解率 (ファイル単位): {file_accuracy:.4f} ({correct_files}/{total_files} ファイル正解)\n")


    #混同行列の作成
    cm = confusion_matrix(y_test_files, first_column_list)
    cm_df = pd.DataFrame(cm, 
                        index=['正解: 0', '正解: 1'], 
                        columns=['予測: 0', '予測: 1'])

    print("混同行列:")
    print(cm_df)
    print("\n") # 見やすいように改行

         AU01      AU02      AU04      AU05      AU06      AU09      AU10  \
0    0.786673  0.585278  0.833798  0.552333  0.281521  0.515034  0.231088   
1    0.845393  0.820645  0.833010  0.720385  0.639244  0.565347  0.868956   
2    0.823720  0.567153  0.818057  0.739919  0.487160  0.496623  0.467757   
3    0.810446  0.634203  0.770241  0.566895  0.447854  0.429799  0.382528   
4    0.832250  0.696070  0.836230  0.748750  0.563139  0.543918  0.543216   
..        ...       ...       ...       ...       ...       ...       ...   
459  0.797877  0.591720  0.545756  0.373633  0.634805  0.412694  0.926595   
460  0.813775  0.651270  0.828008  0.421900  0.747324  0.487996  0.968596   
461  0.785423  0.621684  0.652966  0.357824  0.665490  0.459155  0.942941   
462  0.793317  0.583096  0.614829  0.380039  0.458521  0.487988  0.815109   
463  0.809973  0.641156  0.749186  0.417375  0.250520  0.432180  0.853045   

         AU12      AU14      AU15      AU17      AU23      AU24      AU25  

KeyboardInterrupt: 

#### 学習器をXGBoostからRandomForestに

In [47]:
from sklearn.ensemble import RandomForestClassifier
from boruta import BorutaPy
#学習・テストデータとなるファイルの分割
X_files = file_with_labels_df.drop(columns='Delirum', errors='ignore')
#print(X_files)
y_labels = file_with_labels_df.drop(columns=['filepath', 'filename', 'ID'], errors='ignore')
#rint(y_labels)

#層化５分割交差検証
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(skf.split(X_files, y_labels)):
    X_train_sort, X_test_sort = X_files.iloc[train_index], X_files.iloc[test_index]
    y_train_sort, y_test_sort = y_labels.iloc[train_index], y_labels.iloc[test_index]

    train_file_path = [f for f in X_train_sort['filepath']]
    #print(type(train_file_path), type(y_train_sort['Delirium']))
    test_file_path = [f for f in X_test_sort['filepath']]
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()
    train_file_lengths = []
    test_file_lengths = []
    #print(type(test_file_lengths))

    #学習データ
    train_file_lengths, train_df = answer_max(
        train_file_path,
        y_train_sort['Delirium'],
        train_file_lengths)
    train_df['filenumber'] = train_df['filename'].str.extract(r'(\d+)')
    train_df = train_df.drop(columns=['filename'], errors='ignore')
    #print(train_df)
    #テストデータ
    test_file_lengths, test_df = answer_max(
        test_file_path,
        y_test_sort['Delirium'],
        test_file_lengths
    )

    #グリッドサーチのための学習・検証データの用意とテストデータの用意
    y_train_grid = train_df['Delirium']
    y_test_grid = test_df['Delirium']

    columns_to_delete = ['Delirium']
    columns_to_delete2 = ['Delirium', 'filename']

    X_train_grid = train_df.drop(columns=columns_to_delete, errors='ignore')
    X_test_grid = test_df.drop(columns=columns_to_delete2, errors='ignore')

    #print(X_train_boruta)

    #患者ごとに分割するための変数を用意し、学習データからfilenumberを削除
    patient_groups = X_train_grid['filenumber'].values
    X_train_boruta = X_train_grid.drop(columns=['filenumber']).copy()

    #Add.------学習データだけを使用してBorutaを実行------

    #Borutaの実行
    Rf_boruta = RandomForestClassifier(n_jobs=-1, max_depth=5)
    feat_selector = BorutaPy(Rf_boruta, n_estimators='auto', verbose=2, random_state=1)
    feat_selector.fit(X_train_boruta, y_train_grid)
    X_train_model = X_train_boruta.loc[:, feat_selector.support_]
    X_test_model = X_test_grid.loc[:, feat_selector.support_]

    #Borutaで採用、削除した特徴量の確認
    feature_names = np.array(X_train_boruta.columns)
    dropped_features = feature_names[~feat_selector.support_]
    accepted_features = feature_names[feat_selector.support_]
    tentative_features = feature_names[feat_selector.support_weak_]

    print(f"削除前の特徴量数：{len(feature_names)}")
    print(f"削除された特徴量数：{len(dropped_features)}")
    print("削除された特徴量一覧")
    print(dropped_features)

    #5.------学習の準備(学習器と学習方法の設定)------

    #モデル・グリッドサーチの定義
    model = RandomForestClassifier(random_state=42)
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10, 15],
        'min_samples_split': [2, 5, 10]
        }
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=cv, n_jobs=-1)

    #6.------グリッドサーチ------
    grid_search.fit(X_train_model, y_train_grid, groups=patient_groups)
    print(f"---fold{fold+1}---\n")
    print(f"最適なハイパーパラメータ：{grid_search.best_params_}")
    print(f"CVでのmax精度(Accuracy):{grid_search.best_score_:.4f}")

    #行と列の表示制限を解除（または十分に大きな値に設定）
    pd.set_option('display.max_rows', None)     # すべての行を表示
    pd.set_option('display.max_columns', None)  # すべての列を表示
    pd.set_option('display.width', 1000)        # 表示幅を広くする

    #結果をDataFrameに変換し表示
    results_df = pd.DataFrame(grid_search.cv_results_)

    print("--- グリッドサーチの全試行結果 ---")
    print(results_df)

    #表示設定を元に戻す (推奨)
    #グローバルな設定を元に戻し、他の処理に影響を与えないようにします
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')


    #7.------テストデータでの性能評価------
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_model)
    final_accuracy = accuracy_score(y_test_grid, y_pred)

    print(f"テストデータでの正解率: {final_accuracy:.4}\n")
    #print(y_pred)

    file_predictions = []
    start_idx = 0
    for length in test_file_lengths:
        end_idx = start_idx + length
        predictions_for_this_file = y_pred[start_idx:end_idx]
        majority_vote = mode(predictions_for_this_file)[0]
        file_predictions.append([majority_vote, start_idx, end_idx])
        start_idx = end_idx

    first_column_list = [item[0] for item in file_predictions]

    #print(first_column_list)

    y_test_files = X_test_sort['Delirium'].values

    #print(type(y_test_files))

    y_test_files_copy = X_test_sort

    y_test_files_copy = y_test_files_copy.assign(Predictions=first_column_list)

    print("各テストファイルごとの分類結果\n")
    print(y_test_files_copy.drop(columns=['filepath'], errors='ignore'))

    correct_files = np.sum(np.array(first_column_list) == y_test_files)
    total_files = len(X_test_sort)
    file_accuracy = correct_files/ total_files

    scores = []
    scores.append(file_accuracy)
    print("\n")
    print(f"正解率 (ファイル単位): {file_accuracy:.4f} ({correct_files}/{total_files} ファイル正解)\n")


    #8.------混同行列の作成------
    cm = confusion_matrix(y_test_files, first_column_list)
    cm_df = pd.DataFrame(cm, 
                        index=['正解: 0', '正解: 1'], 
                        columns=['予測: 0', '予測: 1'])

    print("混同行列:")
    print(cm_df)
    print("\n") # 見やすいように改行

    #特徴量の重要度を取得
    feature_names = X_train_model.columns.tolist()
        
    importances = best_model.feature_importances_

    #特徴量の名前と重要度をまとめたDataFrameを作成
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })

    #重要度が高い順にソート
    sorted_importance_df = importance_df.sort_values(by='Importance', ascending=False)

    print(sorted_importance_df)

Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	18
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	9 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	10 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	11 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	12 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	13 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	14 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	15 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration: 	16 / 100
Confirmed: 	4
Tentative: 	6
Rejected: 	8
Iteration:

KeyboardInterrupt: 